## !Pip install

In [44]:
!pip install scikit-learn underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.8/657.8 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.6 MB/s eta 0:00:00


# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Libraries

In [45]:
import pandas as pd
import numpy as np

from underthesea import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer, CountVectorizer
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split

# Đọc dữ liệu

In [25]:
data_path  = "/content/drive/MyDrive/IS353 - Mạng xã hội - Nhóm 6/Đồ án môn học/Project/Data/Data_merge/data_merge.xlsx"

In [26]:
df1 = pd.read_excel(data_path)

In [35]:
df2 = pd.read_excel("/content/drive/MyDrive/IS353 - Mạng xã hội - Nhóm 6/Đồ án môn học/Project/TempCode/courses_describe.xlsx")

In [36]:
df2 = df2[['Mã MH', 'Tóm tắt môn học']]

In [38]:
df2.sample(10)

,Mã MH,Tóm tắt môn học
227,is252,Cung cấp các kiến thức về việc khai thác tri t...
282,msis4523,Bao quát các loại mạng và giao thức mạng được ...
24,ce331,Nội dung môn học này cung cấp cho sinh viên nh...
366,se332,Môn học cung cấp cho sinh viên những kiến thức...
367,se334,"Học phần này trình bày các kiến trúc, nền tảng..."
153,ec208,Trình bày các khía cạnh quan trọng để triển kh...
370,se344,Môn học cung cấp cho sinh viên những kiến thức...
98,cs409,Môn học có nội dung bao gồm 2 phần. Phần lý th...
381,se401,Môn học trình bày các mẫu thiết kế hiện đang đ...
233,is351,Phân tích không gian là một chức năng quan trọ...


In [37]:
# Loại trừ các cột 'mssv'
exclude_columns = ['Tóm tắt môn học']

# Chỉ xử lý các cột kiểu object và không nằm trong exclude_columns
for column in df2.select_dtypes(include=['object']).columns:
    if column not in exclude_columns:
        df2[column] = df2[column].str.lower().str.replace(r"[',:);*@$/%\-]", '', regex=True)

In [39]:
# Ghép hai DataFrame dựa trên cột `mamh` và `Mã MH`
#df = pd.merge(df1, df2, left_on="mamh", right_on="Mã MH", how= 'left')
df = df1.merge(df2, left_on="mamh", right_on="Mã MH", how='left')

In [41]:
df = df.drop(columns=["Mã MH"])

In [42]:
df.sample(10)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,mamta,diem_thita,...,hockyxl,namhocxl,ngayqd,dtb_toankhoa,dtb_tichluy,dtbhk,drl,ghichu,sinhvien_nam,Tóm tắt môn học
68634,E5B910BDXPvAibaEXe/H1VZJBzJAwkG5vgo/54/z,1996,1,quảngngãi,ktmt0001,ktmt,cqui,9,không,0,...,1,2018,12/10/2018,5.95,6.59,3.85,65,Trungbình,2,Môn học cung cấp các kiến thức về các kỹ năng ...
184069,C39EC9E7XPvAibaEXe/d4nFnIhFaQm1KX1sm6B+1,1996,1,lâmđồng,mmtt0001,mmt&tt,cqui,10,không,0,...,1,2020,28/03/2019,7.11,7.43,7.78,68,Khá,3,Môn học hướng về việc trang bị cho sinh viên n...
56571,7DD93E1EXPvAibaEXe8jolcsAIssTCvlawktLHX9,1994,1,thànhphốđànẵng,khmt0001,khmt,cqui,8,không,0,...,1,2019,22/10/2019,6.71,6.98,6.69,48,Yếu,3,Môn học trang bị cho sinh viên tư tưởng Hồ Chí...
304977,EA0847F0XPvAibaEXe+wCRIlhV+1X8xOSYw7pQZ0,2000,0,bìnhđịnh,ktpm2018,cnpm,cqui,13,77,eng02,...,0,0,0,6.95,7.69,6.99,81,Tốt,2,Môn học này trình bày các khái niệm và phương ...
11994,E934BCD3XPvAibaEXe86736b6Ol+/MLJ5vOrqV8U,1994,1,tháibình,khmt0001,khmt,cqui,8,không,0,...,2,2016,18/04/2017,3.75,6.09,3.17,33,Yếu,3,Môn học cung cấp các kiến thức về các kỹ năng ...
25017,B2816B7BXPvAibaEXe/YKN5ysyYL7MS3vlfPnRU+,1995,1,nghệan,ktpm0001,cnpm,cqui,8,không,0,...,2,2016,18/04/2017,4.33,6.39,3.06,70,Khá,2,Môn học sẽ cung cấp các kiến thức nền tảng về ...
206617,175475BBXPvAibaEXe8dBqMaN7rGhkux+ZR35bi9,1998,1,hồ chí minh,httt0001,httt,clc,11,39,en001,...,0,0,0,7.00,7.00,7.30,68,Khá,3,"Cung cấp cho sinh viên những kiến thức, kỹ năn..."
242806,73F641D0XPvAibaEXe/bcEbwNVP7GyCcbmapxqkQ,1999,1,bạcliêu,httt2017,httt,cqui,12,40,eng01,...,0,0,0,8.01,8.01,8.25,100,Xuấtsắc,4,Môn học trình bày các khái niệm cơ bản của điệ...
357600,32947B1DXPvAibaEXe+1xn1m12iBvlw7QT1hKBoM,2001,1,tâyninh,httt2019,httt,cqui,14,285,toeic,...,0,0,0,7.85,7.85,7.09,81,Tốt,1,Môn học cung cấp các kiến thức về các kỹ năng ...
131725,932089EEXPvAibaEXe/Q6C3BndaQULTeezxTyf6z,1997,1,angiang,cntt0001,kttt,cqui,10,không,0,...,0,0,0,8.02,8.02,7.48,85,Tốt,3,"Cung cấp cho sinh viên những kiến thức, kỹ năn..."


# DataMerge

In [ ]:
# Lựa chọn các cột quan trọng
selected_columns = ['mamh', 'diem_hp', 'mssv', 'hocky', 'sinhvien_nam']
data = df[selected_columns]

# Giảm dữ liệu xuống còn 30,000 dòng
data = data.sample(n=30000, random_state=42)

# Giảm chiều dữ liệu và reset chỉ số
data = data.reset_index(drop=True)


In [ ]:
# Making NLP Model

# Tạo vectorizer TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(data['mamh'])

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Giảm chiều dữ liệu và reset chỉ số
train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

# Tạo ma trận TF-IDF cho tập huấn luyện và tập kiểm tra
tfidf_matrix_train = vectorizer.transform(train_data['mamh'])
tfidf_matrix_test = vectorizer.transform(test_data['mamh'])

# Tính ma trận tương đồng cosine cho tập huấn luyện
similarity_matrix_train = cosine_similarity(tfidf_matrix_train)

In [ ]:
data

,mamh,diem_hp,mssv,hocky,sinhvien_nam
0,cs106,3.7,5DB7334BXPvAibaEXe9fafwYRU+Nplkr/FpgEmW3,2,4
1,nt101,2.6,C57530A4XPvAibaEXe8hvnm4ihleLVTcBUD7ztLV,2,3
2,it002,5.0,C1F37059XPvAibaEXe9cYfl6D6BtPG7HzaS511bz,1,2
3,nt204,0.0,49F95988XPvAibaEXe+q0E3QguGBpXAns47p0PqO,2,4
4,it004,1.0,2EB80034XPvAibaEXe/8X6gsb95YiHj38qo0I8/f,1,3
...,...,...,...,...,...
29995,ce103,6.2,A9583138XPvAibaEXe8pd25OrRS+bd8RTXO2y2Hk,1,3
29996,it002,3.8,740A2C5AXPvAibaEXe9NdAclo3+sVy/niSBfKxtQ,2,3
29997,ph002,6.9,C4F596B7XPvAibaEXe9Qk6hDUWKQXZP98Rz5k643,2,1
29998,ma003,8.1,3A8DC76BXPvAibaEXe/waquVAMA8Tk6+UT/Qyju7,1,1


In [ ]:
train_data

,mamh,diem_hp,mssv,hocky,sinhvien_nam
0,it006,6.5,FFEF294AXPvAibaEXe/ceziXFRXnLc/x/K0hVw4d,1,2
1,it009,9.0,AB9BD45CXPvAibaEXe/JKySBFnJ8IhGx+HV+WPqn,2,2
2,ss001,5.8,2EE120A8XPvAibaEXe+tKlY40ATmdSk7aps/Iq2R,1,2
3,it002,0.0,9A84BBE7XPvAibaEXe8ydgsBCmjqA6hvcbpa5SDa,2,1
4,ce117,8.0,4743F40FXPvAibaEXe+/MvKIb2VAVUWMGXsSclXS,1,3
...,...,...,...,...,...
23995,ss004,0.0,8B815297XPvAibaEXe//3t8+us0X0khfBNlWfzpk,1,2
23996,ma003,7.1,383910F9XPvAibaEXe+7yjglV2dMp44sgNgpkvEP,1,1
23997,ie105,8.1,EF3FEDEFXPvAibaEXe/57ZZLpsI5pRuXtM45WaKF,1,4
23998,it007,3.0,5DB7334BXPvAibaEXe9fafwYRU+Nplkr/FpgEmW3,2,3


In [ ]:
test_data

,mamh,diem_hp,mssv,hocky,sinhvien_nam
0,ma002,4.5,6426D277XPvAibaEXe+kwIr/VXjl04GQLN5sEC66,2,1
1,ma005,5.2,11D7FBB7XPvAibaEXe8/3iNqySORby7kyf2IzNeI,2,2
2,en002,7.2,5F66CC15XPvAibaEXe/aPLdUCtOgCmaknvOevy6n,1,2
3,nt114,5.0,E90E1277XPvAibaEXe/wM5TrBBCDMYjuhhJRnFGL,2,3
4,ss003,6.4,5922FCABXPvAibaEXe8AEUw13Ud6KnHmyBmlVUoY,1,3
...,...,...,...,...,...
5995,pe002,7.0,B72FC904XPvAibaEXe+oM1KMvbbnDO/T+VJFcoqm,2,1
5996,it009,6.8,19B4EB0FXPvAibaEXe+pDN88/1DR4JOjz3aukUSu,1,2
5997,ma003,9.0,12A164C7XPvAibaEXe9eYPx3anXXhP21P/VRYCGB,1,1
5998,se358,9.0,99092200XPvAibaEXe9OPLPOPVklDFoc2adYJo3E,1,4


In [ ]:
# Gợi ý các môn học liên quan
def recommend_courses(course_id, similarity_matrix, data, top_n=5):
    if course_id in data['mamh'].values:
        course_index = data[data['mamh'] == course_id].index[0]
        similarity_scores = list(enumerate(similarity_matrix[course_index]))
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
        top_courses = [data.iloc[score[0]]['mamh'] for score in similarity_scores]

        # Loại bỏ môn học đang xét và các môn học bị trùng
        top_courses = list(set(top_courses) - {course_id})

        return top_courses[:top_n]
    else:
        return []

In [ ]:
# Gợi ý các môn học liên quan đến môn 'se358'
recommended_courses = recommend_courses('cs106', similarity_matrix_train, train_data)
print("Các môn học liên quan đến 'SS004':")
for course in recommended_courses:
    print(course)

Các môn học liên quan đến 'SS004':
nt538
se350
is502
ds307
cs551


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_cols = ['hedt', 'xeploai']
for col in label_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

In [ ]:
student_features = df[['mssv', 'gioitinh', 'khoa', 'hedt', 'dtb_tichluy', 'drl']].drop_duplicates()

In [ ]:
course_features = df[['mamh', 'sotc']].drop_duplicates()

In [ ]:
interactions = df[['mssv', 'mamh', 'diem_hp']]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

course_similarity = cosine_similarity(course_features.drop('mamh', axis=1))

In [ ]:
duplicates = interactions[interactions.duplicated(subset=['mssv', 'mamh'], keep=False)]
print(duplicates)
interactions = interactions.groupby(['mssv', 'mamh'], as_index=False).agg({'diem_hp': 'mean'})


                                            mssv   mamh  diem_hp
5       BE375BAAXPvAibaEXe9JDlHA4z2GHJ3/PVStCxR2  ss001      4.5
13      BE375BAAXPvAibaEXe9JDlHA4z2GHJ3/PVStCxR2  ss001      4.5
27      BE375BAAXPvAibaEXe9JDlHA4z2GHJ3/PVStCxR2  se102      2.5
28      BE375BAAXPvAibaEXe9JDlHA4z2GHJ3/PVStCxR2  ss001      6.4
37      BE375BAAXPvAibaEXe9JDlHA4z2GHJ3/PVStCxR2  se215      3.5
...                                          ...    ...      ...
425648  7D7633B0XPvAibaEXe95IhrkYSGfcCQVGm4nFGyt  nt106      0.0
425732  CB263C18XPvAibaEXe8xMCTZ03/Be8yk40QWdPiR  it004      4.3
425758  CB263C18XPvAibaEXe8xMCTZ03/Be8yk40QWdPiR  it004      7.8
425767  7CD9B404XPvAibaEXe8M4ltS8XjC9c/g97NvpP52  ma003      4.8
425784  7CD9B404XPvAibaEXe8M4ltS8XjC9c/g97NvpP52  ma003      7.2

[146739 rows x 3 columns]


In [ ]:
# Tạo ma trận tương tác
interaction_matrix = interactions.pivot(index='mssv', columns='mamh', values='diem_hp').fillna(0)

# Đồng bộ kích thước giữa interaction_matrix và course_similarity
interaction_matrix = interaction_matrix.reindex(columns=course_features['mamh'], fill_value=0)

# Tính điểm dự đoán
predicted_scores = np.dot(interaction_matrix.values, course_similarity)

# Đưa ra gợi ý môn học
top_recommendations = np.argsort(-predicted_scores, axis=1)


In [ ]:
# Tạo ma trận thực tế
actual_scores_matrix = interactions.pivot(index='mssv', columns='mamh', values='diem_hp').fillna(0)
actual_scores_matrix = actual_scores_matrix.reindex(columns=interaction_matrix.columns, fill_value=0)
actual_scores = actual_scores_matrix.values

# Tính RMSE
rmse = np.sqrt(mean_squared_error(actual_scores, predicted_scores))
print(f'RMSE: {rmse}')

RMSE: 362.38680296306336


In [ ]:
def recommend_courses(student_id, interaction_matrix, predicted_scores, course_features, top_n=5):
    # Lấy index của sinh viên trong ma trận tương tác
    if student_id not in interaction_matrix.index:
        return f"Sinh viên {student_id} không có trong dữ liệu."

    student_idx = interaction_matrix.index.get_loc(student_id)

    # Lấy điểm dự đoán cho sinh viên đó
    student_scores = predicted_scores[student_idx]

    # Sắp xếp môn học theo điểm dự đoán giảm dần
    recommended_indices = np.argsort(-student_scores)

    # Lọc các môn học mà sinh viên chưa học
    student_courses = interaction_matrix.loc[student_id]
    unlearned_courses = [
        interaction_matrix.columns[i]
        for i in recommended_indices
        if student_courses.iloc[i] == 0
    ]

    # Trả về danh sách các môn học được gợi ý
    recommended_courses = course_features[course_features['mamh'].isin(unlearned_courses[:top_n])]
    return recommended_courses[['mamh', 'sotc']]


In [ ]:
# Mã sinh viên cần thử gợi ý
student_id = 'BE375BAAXPvAibaEXe9JDlHA4z2GHJ3/PVStCxR2'  # Thay bằng mã sinh viên bạn muốn thử

# Gợi ý môn học
recommendations = recommend_courses(
    student_id,
    interaction_matrix,
    predicted_scores,
    course_features,
    top_n=5
)

print("Môn học được đề xuất:")
print(recommendations)


Môn học được đề xuất:
         mamh  sotc
57050   cs526     4
62423   eng03     4
62832   ie307     4
66497   cs529     4
66505   cs338     4
66543   cs526     3
184277  eng03     0


In [ ]:
from sklearn.preprocessing import LabelEncoder
label_cols = ['gioitinh', 'hedt', 'khoa']
for col in label_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

In [ ]:
# Xử lý dữ liệu sinh viên và môn học
student_features = df[['mssv', 'khoa', 'sinhvien_nam', 'gioitinh', 'dtb_tichluy', 'dtb_toankhoa', 'drl']].drop_duplicates()
course_features = df[['mamh', 'sotc', 'khoahoc', 'mamta']].drop_duplicates()

# Kiểm tra và xử lý dữ liệu trùng lặp
duplicates = df[df.duplicated(subset=['mssv', 'mamh'], keep=False)]
print("Số lượng bản ghi trùng lặp:", len(duplicates))

# Gộp các giá trị trùng lặp bằng cách lấy trung bình điểm
df = df.groupby(['mssv', 'mamh'], as_index=False).agg({'diem_hp': 'mean'})

# Tạo ma trận tương tác
interactions = df.pivot(index='mssv', columns='mamh', values='diem_hp').fillna(0)
print("Ma trận tương tác đã tạo thành công.")


Số lượng bản ghi trùng lặp: 146739
Ma trận tương tác đã tạo thành công.


In [ ]:
# Xác định các cột có chứa dữ liệu chuỗi hoặc hỗn hợp
string_columns = course_features.select_dtypes(include=['object', 'int', 'float']).columns

# Chuyển đổi tất cả giá trị trong cột thành chuỗi
for col in string_columns:
    course_features[col] = course_features[col].astype(str)



In [ ]:
# Áp dụng Label Encoding cho các cột
from sklearn.preprocessing import LabelEncoder
for col in string_columns:
    le = LabelEncoder()
    course_features[col] = le.fit_transform(course_features[col])



In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# ===== 1. Giả sử dữ liệu đã được nạp vào DataFrame 'course_features' =====
# course_features chứa các cột: 'mamh', 'sotc', 'khoahoc', 'mamta'

# Kiểm tra thông tin dữ liệu
print(course_features.info())
print(course_features.describe())

# ===== 2. Loại bỏ cột không cần thiết =====
reduced_course_features = course_features.drop(['mamh'], axis=1)

# ===== 3. Kiểm tra số chiều của dữ liệu =====
n_features = reduced_course_features.shape[1]

# Áp dụng PCA nếu số chiều > 2
if n_features > 2:
    n_components = min(n_features, 10)  # Giảm chiều không vượt quá 10 hoặc số chiều hiện tại
    pca = PCA(n_components=n_components)
    reduced_features = pca.fit_transform(reduced_course_features)
    print(f"Áp dụng PCA thành công: Dữ liệu giảm xuống {reduced_features.shape[1]} chiều.")
else:
    reduced_features = reduced_course_features.values
    print(f"Dữ liệu có {n_features} chiều, không cần PCA.")

# ===== 4. Tính toán điểm tương đồng theo từng batch =====
def compute_similarity_in_batches(features, batch_size=500):
    n = features.shape[0]
    similarity_matrix = np.zeros((n, n))  # Khởi tạo ma trận tương đồng

    for i in range(0, n, batch_size):
        end_i = min(i + batch_size, n)
        batch = features[i:end_i]  # Chọn batch hiện tại
        similarity_matrix[i:end_i] = cosine_similarity(batch, features)  # Tính cosine similarity

        print(f"Đã xử lý batch từ {i} đến {end_i}/{n}")

    return similarity_matrix

# Tính toán ma trận tương đồng theo batch
course_similarity = compute_similarity_in_batches(reduced_features, batch_size=1000)
print("Tính toán ma trận tương đồng giữa các khóa học thành công!")

# ===== 5. Kiểm tra kết quả =====
# Kích thước của ma trận tương đồng
print(f"Kích thước ma trận tương đồng: {course_similarity.shape}")

# Hiển thị một phần ma trận tương đồng
print("5x5 phần tử đầu của ma trận tương đồng:")
print(course_similarity[:5, :5])


<class 'pandas.core.frame.DataFrame'>
Index: 50179 entries, 0 to 425799
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   mamh     50179 non-null  int64
 1   sotc     50179 non-null  int64
 2   khoahoc  50179 non-null  int64
 3   mamta    50179 non-null  int64
dtypes: int64(4)
memory usage: 1.9 MB
None
               mamh          sotc       khoahoc         mamta
count  50179.000000  50179.000000  50179.000000  50179.000000
mean     236.428466      4.183623      2.707527    116.970585
std      116.639706      1.122762      1.157921     54.368389
min        0.000000      0.000000      0.000000      0.000000
25%      156.000000      4.000000      2.000000     73.000000
50%      248.000000      4.000000      3.000000    118.000000
75%      338.000000      5.000000      4.000000    166.000000
max      420.000000      6.000000      6.000000    204.000000
Áp dụng PCA thành công: Dữ liệu giảm xuống 3 chiều.
Đã xử lý batch từ 0 đến 100

In [51]:
# Giả sử df là DataFrame của bạn
# Xóa các dòng có giá trị NaN trong cột "Tóm tắt môn học"
df = df.dropna(subset=['Tóm tắt môn học'])

# Hiển thị DataFrame sau khi xóa
print("DataFrame sau khi xóa các dòng có giá trị NaN trong cột 'Tóm tắt môn học':")

DataFrame sau khi xóa các dòng có giá trị NaN trong cột 'Tóm tắt môn học':


In [54]:
# Hàm tiền xử lý: tách từ và chuẩn hóa
def preprocess_text(text):
    # Tách từ bằng Underthesea
    tokenized_text = word_tokenize(text, format="text")
    return tokenized_text

# Tiền xử lý dữ liệu
processed_courses_describe = [preprocess_text(text) for text in df['Tóm tắt môn học']]

# Tạo TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Chuyển đổi văn bản sang ma trận TF-IDF
tfidf_matrix = vectorizer.fit_transform(processed_courses_describe)

# Hiển thị các từ vựng và trọng số TF-IDF
vocab = vectorizer.get_feature_names_out()
print("Từ vựng:", vocab)
print("Ma trận TF-IDF:")
print(tfidf_matrix.toarray())

Từ vựng: ['01' '03' '06' ... 'ứng_dụng_cụ_thể' 'ứng_phó' 'ứng_xử']
Ma trận TF-IDF:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [ ]:
indices = pd.Series(df.index, index=df['mamh']).drop_duplicates()
indices